<a href="https://colab.research.google.com/github/MawaSC/Quantum-Simulation-on-IQM-Resonance/blob/main/03_noise_simulation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

!pip install qiskit-aer

from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel
from qiskit_aer.noise import depolarizing_error

from fermi_hubbard_model import (
    L,
    occupations_from_counts
)

1. Simulate depolarizing gate noise

We compare an ideal circuit simulation with one that includes depolarizing noise. This lets us see how the noise model affects the results.

We choose the noise settings ourselves, so they should not be treated as the actual error rates of IQM hardware. We also run the noisy simulation with several random seeds to see how much the measured occupations vary between runs.

This experiment shows what happens under our chosen noise settings. It does not predict IQM hardware performance unless the settings are supported by actual device data.

In [ ]:
qc = qv.qs.to_qiskit()
qc.measure_all()

ideal_sim = AerSimulator()

error1_qubit = depolarizing_error(0.001, 1)

error2_qubit = depolarizing_error(0.001, 2)

noise_model = NoiseModel()

noise_model.add_all_qubit_quantum_error(error1_qubit, ['rx', 'ry', 'rz', 'h', 'x'])

noise_model.add_all_qubit_quantum_error(error2_qubit, ['cx', 'cz', 'swap'])

noisy_sim = AerSimulator(noise_model=noise_model)

In [ ]:
seeds = [1234, 5678, 9012, 3456, 7890]

noisy_up_runs = []
noisy_down_runs = []
noisy_total_runs = []

for seed in seeds:

  noisy_result = noisy_sim.run(qc, shots=4096, seed_simulator=seed).result()

  noisy_counts = noisy_result.get_counts()

  noisy_up, noisy_down = occupations_from_counts(noisy_counts, flipped=False)

  noisy_total = noisy_up + noisy_down

  noisy_up_runs.append(noisy_up)
  noisy_down_runs.append(noisy_down)
  noisy_total_runs.append(noisy_total)


ideal_result = ideal_sim.run(qc, shots=4096, seed_simulator=1234).result()


ideal_counts = ideal_result.get_counts()

In [ ]:
ideal_up, ideal_down = occupations_from_counts(ideal_counts, flipped=False)

In [ ]:
noisy_up_runs = np.array(noisy_up_runs)
noisy_down_runs = np.array(noisy_down_runs)
noisy_total_runs = np.array(noisy_total_runs)

In [ ]:
mean_up = np.mean(noisy_up_runs, axis=0)
std_up = np.std(noisy_up_runs, axis=0, ddof=1)

mean_down = np.mean(noisy_down_runs, axis=0)
std_down = np.std(noisy_down_runs, axis=0, ddof=1)

mean_total = np.mean(noisy_total_runs, axis=0)
std_total = np.std(noisy_total_runs, axis=0, ddof=1)

In [ ]:
plt.figure(figsize=(8,5))

plt.plot(
    range(L),
    ideal_up,
    'o-',
    linewidth=2,
    label="Ideal Simulation"
)

plt.plot(
    range(L),
    mean_up,
    's--',
    linewidth=2,
    label="Depolarizing Noise (Mean)"
)

plt.xlabel("Lattice Site")
plt.ylabel("Spin-Up Occupation")
plt.title("Ideal Simulation vs Depolarizing Noise (Spin-Up)")
plt.legend()

plt.grid()

# 1. Save as PNG (dpi=300 for crisp quality, bbox_inches='tight' prevents edge cropping)
plt.savefig('spin_up_occupation_plot.png', bbox_inches='tight', dpi=300)
plt.savefig('spin_up_occupation_plot.pdf', bbox_inches='tight', dpi=300)

plt.show()

In [ ]:
plt.figure(figsize=(8,5))

plt.plot(
    range(L),
    ideal_down,
    'o-',
    linewidth=2,
    label="Ideal Simulation"
)

plt.plot(
    range(L),
    mean_down,
    's--',
    linewidth=2,
    label="Depolarizing Noise (Mean)"
)

plt.xlabel("Lattice Site")
plt.ylabel("Spin-Down Occupation")
plt.title("Ideal Simulation vs Depolarizing Noise (Spin-Down)")
plt.legend()

plt.grid()

# 1. Save as PNG (dpi=300 for crisp quality, bbox_inches='tight' prevents edge cropping)
plt.savefig('spin_down_occupation_plot.png', bbox_inches='tight', dpi=300)
plt.savefig('spin_down_occupation_plot.pdf', bbox_inches='tight', dpi=300)

plt.show()

In [ ]:
ideal_total = ideal_up + ideal_down
noisy_total = noisy_up + noisy_down

plt.figure(figsize=(8,5))

plt.plot(
    range(L),
    ideal_total,
    'o-',
    linewidth=2,
    label="Ideal Simulation"
)

plt.plot(
    range(L),
    mean_total,
    's--',
    linewidth=2,
    label="Depolarizing Noise (Mean)"
)

plt.xlabel("Lattice Site")
plt.ylabel("Total Occupation")
plt.title("Ideal Simulation vs Depolarizing Noise")
plt.legend()

plt.grid()


# 1. Save as PNG (dpi=300 for crisp quality, bbox_inches='tight' prevents edge cropping)
plt.savefig('ideal_vs_depolarizing_comb_plot.png', bbox_inches='tight', dpi=300)
plt.savefig('ideal_vs_depolarizing_comb_plot.pdf', bbox_inches='tight', dpi=300)

plt.show()

In [ ]:
sites = np.arange(L)

plt.figure(figsize=(8,5))

plt.errorbar(
    sites,
    mean_total,
    yerr=std_total,
    fmt='o-',
    capsize=4,
    linewidth=2,
    label="Depolarizing Noise"
)

plt.xlabel("Lattice Site")
plt.ylabel("Total Occupation")
plt.title("Total Site Occuparion Under Depolarizing Noise")

plt.xticks(sites)
plt.legend()

plt.grid()

# 1. Save as PNG (dpi=300 for crisp quality, bbox_inches='tight' prevents edge cropping)
plt.savefig('depolarizing_SD_plot.png', bbox_inches='tight', dpi=300)
plt.savefig('depolarizing_SD_plot.pdf', bbox_inches='tight', dpi=300)

plt.show()

In [ ]:
ideal_total = ideal_up + ideal_down

plt.figure(figsize=(8,5))

plt.plot(
    sites,
    ideal_total,
    'o-',
    linewidth=2,
    label="Ideal Simulation"
)

plt.errorbar(
    sites,
    mean_total,
    yerr=std_total,
    fmt='s--',
    capsize=4,
    linewidth=2,
    label="Depolarizing Noise (mean ± SD)"
)

plt.xlabel("Lattice Site")
plt.ylabel("Total Occupation")
plt.title("Ideal Simulation vs Depolarizing Noise")

plt.xticks(sites)
plt.legend()

plt.grid()

# 1. Save as PNG (dpi=300 for crisp quality, bbox_inches='tight' prevents edge cropping)
plt.savefig('ideal_vs_depolarizing_SD_plot.png', bbox_inches='tight', dpi=300)
plt.savefig('ideal_vs_depolarizing_SD_plot.pdf', bbox_inches='tight', dpi=300)

plt.show()

In [ ]:
results_table = pd.DataFrame({
    "Lattice Site": sites,
    "Ideal Occupation": ideal_total,
    "Noisy Mean": mean_total,
    "Noise SD": std_total
})

results_table = results_table.round({
    "Ideal Occupation": 3,
    "Noisy Mean": 3,
    "Noise SD": 3
})

print(results_table.to_string(index=False))

In [ ]:
# 1. Set up figure
fig, ax = plt.subplots(
    figsize=(7, len(results_table) * 0.5 + 1),
    dpi=300
)

ax.axis('off')

# 2. Create table
table = ax.table(
    cellText=results_table.values,
    colLabels=results_table.columns,
    cellLoc='center',
    loc='center'
)

# 3. Format text and spacing
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1.2, 1.8)

# 4. Style table
header_color = '#1f4e78'
zebra_color = '#f2f6fa'

for (row, col), cell in table.get_celld().items():
    cell.set_edgecolor('#d0d7de')

    if row == 0:
        # Header
        cell.set_facecolor(header_color)
        cell.get_text().set_color('white')
        cell.get_text().set_weight('bold')
    else:
        # Alternating row colors
        cell.set_facecolor(
            zebra_color if row % 2 == 0 else '#ffffff'
        )
        cell.get_text().set_color('#222222')

# 5. Save as PNG
plt.savefig(
    'depolarizing_noise_results_table.png',
    bbox_inches='tight',
    dpi=300
)

# 6. Save as PDF
plt.savefig(
    'depolarizing_noise_results_table.pdf',
    bbox_inches='tight',
    dpi=300
)

plt.close()

print("Table saved as PNG and PDF.")

In [ ]:
print(qv.qs.data)
print(type(qv.qs))
print(dir(qv.qs))